In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [2]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import json

In [3]:
# Re-load CSV with 'location' included in metadata
loader = CSVLoader(
    file_path="/kaggle/input/datasets/ahmedayman7/locations-compounds-and-blogs-data/locations_compounds_and_blogs_data.csv",
    source_column="content",
    metadata_columns=["url"]
)
docs = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Max characters per chunk
    chunk_overlap=200,   # Overlap for context
    add_start_index=True # Track original position
)
chunks = splitter.split_documents(docs)

In [5]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [6]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./locations_compounds_and_blogs_db"  # Saves locally
)

In [7]:
!zip -r locations_compounds_and_blogs_db.zip locations_compounds_and_blogs_db

  adding: locations_compounds_and_blogs_db/ (stored 0%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/ (stored 0%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/header.bin (deflated 55%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/data_level0.bin (deflated 53%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/index_metadata.pickle (deflated 44%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/length.bin (deflated 97%)
  adding: locations_compounds_and_blogs_db/33bfd717-7072-4474-96cc-83b2abd76b86/link_lists.bin (deflated 77%)
  adding: locations_compounds_and_blogs_db/chroma.sqlite3 (deflated 78%)
